# Notebook 3: Baseline Models
**PURPOSE:** Train all 6 models with default parameters. Find top performers.

In [1]:
import os
project_root = r"C:\Users\srich\OneDrive\Desktop\prediction-model"
os.chdir(project_root)
print(f"✅ Working directory: {os.getcwd()}")

✅ Working directory: C:\Users\srich\OneDrive\Desktop\prediction-model


In [2]:
import pandas as pd
import numpy as np
import pickle, json, time, warnings, subprocess
warnings.filterwarnings('ignore')

for pkg in ['xgboost', 'lightgbm', 'catboost']:
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.run(['pip', 'install', pkg, '--quiet'], check=True)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score, log_loss, classification_report
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

print('All libraries loaded ✅')

All libraries loaded ✅


## 3.1 Load Processed Data

In [3]:
train_df = pd.read_csv('data/processed/training_data.csv')
test_df = pd.read_csv('data/processed/test_data.csv')
with open('model/feature_names.json') as f:
    feature_cols = json.load(f)

X_train = train_df[feature_cols]
y_train = train_df['result']
X_test = test_df[feature_cols]
y_test = test_df['result']

print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'Features: {len(feature_cols)}')

X_train: (379184, 28)
X_test:  (19410, 28)
Features: 28


## 3.2 Define All 6 Models

In [4]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42, n_jobs=-1),
    'Random Forest': RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=42, n_jobs=-1, verbosity=0),
    'LightGBM': LGBMClassifier(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1),
    'CatBoost': CatBoostClassifier(iterations=100, random_seed=42, verbose=0),
    'MLP Neural Network': MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=500, random_state=42, early_stopping=True, validation_fraction=0.1)
}
print(f'Defined {len(models)} models ✅')

Defined 6 models ✅


## 3.3 Evaluation Function

In [5]:
def evaluate_model(model, X_test, y_test, model_name, train_time):
    start = time.time()
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    pred_time = (time.time() - start) * 1000

    acc = accuracy_score(y_test, y_pred)
    brier = brier_score_loss(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ll = log_loss(y_test, y_prob)

    print(f'\n{"="*45}')
    print(f'  {model_name}')
    print(f'{"="*45}')
    print(f'  Accuracy:    {acc:.4f} ({acc*100:.1f}%)')
    print(f'  Brier Score: {brier:.4f} ← MOST IMPORTANT')
    print(f'  ROC-AUC:     {auc:.4f}')
    print(f'  Log Loss:    {ll:.4f}')
    print(f'  Train time:  {train_time:.1f}s')
    print(f'  Pred time:   {pred_time:.0f}ms/1000')

    return {'model': model_name, 'accuracy': acc, 'brier': brier, 'roc_auc': auc, 'log_loss': ll, 'train_time': train_time, 'pred_ms': pred_time}

print('Evaluation function defined ✅')

Evaluation function defined ✅


## 3.4 Train and Evaluate All Models
**Note:** This may take 10-20 minutes.

In [6]:
results = []
trained_models = {}

for name, model in models.items():
    print(f'\nTraining: {name}...')
    start = time.time()
    try:
        model.fit(X_train, y_train)
        train_time = time.time() - start
        result = evaluate_model(model, X_test, y_test, name, train_time)
        results.append(result)
        trained_models[name] = model
    except Exception as e:
        print(f'❌ Error: {e}')
        continue

print('\n\n🏁 ALL MODELS TRAINED!')


Training: Logistic Regression...

  Logistic Regression
  Accuracy:    0.7992 (79.9%)
  Brier Score: 0.1391 ← MOST IMPORTANT
  ROC-AUC:     0.8897
  Log Loss:    0.4341
  Train time:  28.5s
  Pred time:   44ms/1000

Training: Random Forest...

  Random Forest
  Accuracy:    0.7651 (76.5%)
  Brier Score: 0.1566 ← MOST IMPORTANT
  ROC-AUC:     0.8637
  Log Loss:    0.5446
  Train time:  10.2s
  Pred time:   111ms/1000

Training: XGBoost...

  XGBoost
  Accuracy:    0.7377 (73.8%)
  Brier Score: 0.1863 ← MOST IMPORTANT
  ROC-AUC:     0.8212
  Log Loss:    0.6093
  Train time:  2.9s
  Pred time:   17ms/1000

Training: LightGBM...

  LightGBM
  Accuracy:    0.7859 (78.6%)
  Brier Score: 0.1488 ← MOST IMPORTANT
  ROC-AUC:     0.8690
  Log Loss:    0.4595
  Train time:  1.3s
  Pred time:   29ms/1000

Training: CatBoost...

  CatBoost
  Accuracy:    0.7431 (74.3%)
  Brier Score: 0.1887 ← MOST IMPORTANT
  ROC-AUC:     0.8151
  Log Loss:    0.6168
  Train time:  2.8s
  Pred time:   13ms/1000

T

## 3.5 Results Summary Table

In [7]:
results_df = pd.DataFrame(results).sort_values('brier')

print('\n=== FINAL RANKINGS ===')
print('(Sorted by Brier Score — lower better)\n')
print(results_df[['model', 'accuracy', 'brier', 'roc_auc']].to_string(index=False))

print(f'\n🏆 Best model: {results_df.iloc[0]["model"]}')
print(f'   Brier: {results_df.iloc[0]["brier"]:.4f}')
print(f'   Accuracy: {results_df.iloc[0]["accuracy"]:.4f}')

results_df.to_csv('data/processed/baseline_results.csv', index=False)
print('\n✅ Results saved')


=== FINAL RANKINGS ===
(Sorted by Brier Score — lower better)

              model  accuracy    brier  roc_auc
Logistic Regression  0.799176 0.139101 0.889723
           LightGBM  0.785935 0.148801 0.869043
      Random Forest  0.765121 0.156628 0.863685
            XGBoost  0.737713 0.186313 0.821163
           CatBoost  0.743071 0.188667 0.815111
 MLP Neural Network  0.700876 0.261140 0.788233

🏆 Best model: Logistic Regression
   Brier: 0.1391
   Accuracy: 0.7992

✅ Results saved


## 3.6 Phase-Specific Evaluation

In [8]:
phases = {
    'Powerplay (1-6)': test_df['over_number'] <= 6,
    'Middle (7-15)': (test_df['over_number'] > 6) & (test_df['over_number'] <= 15),
    'Death (16-20)': test_df['over_number'] > 15
}

print('=== ACCURACY BY MATCH PHASE ===\n')
best_model_name = results_df.iloc[0]['model']
best_model = trained_models[best_model_name]

for phase_name, mask in phases.items():
    X_phase = X_test[mask]
    y_phase = y_test[mask]
    if len(X_phase) == 0: continue
    y_pred = best_model.predict(X_phase)
    acc = accuracy_score(y_phase, y_pred)
    print(f'{phase_name}: {acc:.1%} ({len(X_phase):,} balls)')

top4 = results_df.head(4)['model'].tolist()
print(f'\n📋 Top 4 for tuning: {top4}')

with open('model/top4_models.json', 'w') as f:
    json.dump(top4, f)

import os
os.makedirs('model', exist_ok=True)
for name, model in trained_models.items():
    safe_name = name.lower().replace(' ', '_')
    with open(f'model/baseline_{safe_name}.pkl', 'wb') as f:
        pickle.dump(model, f)
print('✅ Models saved')

=== ACCURACY BY MATCH PHASE ===

Powerplay (1-6): 71.9% (2,617 balls)
Middle (7-15): 78.3% (11,609 balls)
Death (16-20): 87.5% (5,184 balls)

📋 Top 4 for tuning: ['Logistic Regression', 'LightGBM', 'Random Forest', 'XGBoost']
✅ Models saved
